<a href="https://colab.research.google.com/github/harsha-suribhatla/promptengineeringassignment/blob/main/Case2ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Exercise 2 — Code Generation with ReAct Prompting

Tool used: Google Colab + built-in Gemini (google.colab.ai, no API key)

PROMPT 1 — Code-generation prompt (typed into Gemini in Colab to build the loop)
Write Python code for Google Colab that runs a ReAct loop with google.colab.ai.generate_text(). Send the ReAct prompt below to the model and pull the Python code block out of its response. Run the code with exec() inside try/except and capture the output or the full traceback. If it fails, send the error back to the model as an "Observation" and ask it to fix the code. Stop after it succeeds or after 3 attempts. Print every Thought, Code, and Observation, labeled by iteration number.

PROMPT 2 — ReAct prompt v1 (initial)
You are a Python developer using the ReAct method. Work in these stages and label each one:
Thought: Restate the task, list the inputs and outputs, and plan your approach and edge cases.
Action: Write the complete Python code in one python code block.
Observation: (I will run your code and paste the result or the error here.)
Fix: If the observation shows an error or wrong output, explain the cause, then give corrected code.

Task: Write a function monthly_spend(transactions) that takes a list of dicts with keys "date" (YYYY-MM-DD), "category", and "amount", and returns a dict of total spend per month per category, rounded to 2 decimals.

Requirements:
- Python 3, standard library only (no pandas).
- Skip records with a missing or invalid date or amount, and count how many were skipped.
- Handle an empty list by returning {}.
- Include 4 test cases with asserts: normal data, an empty list, a bad date, and a negative refund amount.
- Print "ALL TESTS PASSED" at the end.

PROMPT 3 — ReAct prompt v2 (final, used in the notebook)
You are a Python developer using the ReAct method. Work in these labeled stages:
Thought: Restate the task, list inputs/outputs, plan the approach and edge cases.
Action: Write the complete Python code in ONE python code block.
Observation: (I will run your code and paste the result or error here.)
Fix: If the observation shows an error or wrong output, explain the cause and give corrected code.

Task: Write a function monthly_spend(transactions) that takes a list of dicts with keys "date" (YYYY-MM-DD), "category", and "amount", and returns a dict of total spend per month per category, rounded to 2 decimals.

Requirements:
- Python 3, standard library only (no pandas).
- Skip records with missing/invalid date or amount; count how many were skipped.
- Accept amounts given as strings like "12.50".
- Empty list returns {}.
- Include 5 assert-based tests: normal data, empty list, bad date, negative refund, string amount.
- Print "ALL TESTS PASSED" at the end.

PROMPT 4 — Fix prompt (sent automatically by the loop only if an attempt fails)
{previous model response}
Observation: {output or full error traceback}
Now give the Fix stage with corrected code.

ITERATION NOTES
- v1 to v2: v1 asked for only 4 tests and didn't cover string amounts. v2 adds "Accept amounts given as strings like '12.50'" and a 5th test so the model has to handle a real-world data-quality edge case.
- Result: With v2, the model passed all 5 tests on attempt 1. Its Thought stage also found extra edge cases I hadn't asked for (missing category, NaN/Inf amounts, non-dict records).
- Fix loop: The loop feeds any error traceback back to the model as an Observation and asks for a Fix, for up to 3 attempts. It wasn't triggered because attempt 1 passed.



In [3]:
from google.colab import ai
import re, io, traceback, contextlib

REACT_PROMPT = """You are a Python developer using the ReAct method. Work in these labeled stages:
Thought: Restate the task, list inputs/outputs, plan the approach and edge cases.
Action: Write the complete Python code in ONE ```python block.
Observation: (I will run your code and paste the result or error here.)
Fix: If the observation shows an error or wrong output, explain the cause and give corrected code.

Task: Write a function monthly_spend(transactions) that takes a list of dicts with keys
"date" (YYYY-MM-DD), "category", and "amount", and returns a dict of total spend per
month per category, rounded to 2 decimals.

Requirements:
- Python 3, standard library only (no pandas).
- Skip records with missing/invalid date or amount; count how many were skipped.
- Accept amounts given as strings like "12.50".
- Empty list returns {}.
- Include 5 assert-based tests: normal data, empty list, bad date, negative refund, string amount.
- Print "ALL TESTS PASSED" at the end.
"""

### ReAct Loop Implementation

This section implements a ReAct loop to iteratively work with the Gemini model. The model will generate Python code based on a task, which will then be executed. If errors occur, the error message will be fed back to the model for correction, up to 3 attempts.


In [5]:
def extract_code(text):
    m = re.findall(r"```python\s*(.*?)```", text, re.DOTALL)
    return m[-1] if m else None

def run_code(code):
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return True, buf.getvalue()
    except Exception:
        return False, buf.getvalue() + "\n" + traceback.format_exc()

conversation = REACT_PROMPT
for attempt in range(1, 4):
    print(f"\n{'='*20} ATTEMPT {attempt} {'='*20}")
    response = ai.generate_text(conversation)
    print(response)

    code = extract_code(response)
    if not code:
        conversation += "\n\nObservation: No ```python code block found. Provide the code."
        continue

    ok, observation = run_code(code)
    print(f"\n--- Observation (attempt {attempt}) ---\n{observation}")

    if ok and "ALL TESTS PASSED" in observation:
        print(f"\n✅ Success on attempt {attempt}")
        break
    conversation += f"\n\n{response}\n\nObservation: {observation}\nNow give the Fix stage with corrected code."
else:
    print("\n❌ Did not pass after 3 attempts")


==================== ATTEMPT 1 ====================
Thought:
I need to write a function `monthly_spend(transactions)` that aggregates spend per month per category.
Inputs: A list of transaction dicts, each with "date", "category", and "amount".
Outputs: A dict of the form `{"YYYY-MM": {"category": total_amount}}` rounded to 2 decimals.
Edge cases:
- Missing/invalid date (must be YYYY-MM-DD) -> skip and increment counter.
- Missing/invalid amount (must be convertible to float, and not NaN/Inf) -> skip and increment counter.
- Missing/invalid category -> skip and increment counter.
- Empty list input -> return `{}`.
- Negative amounts (refunds) -> allowed.
- Keep track of the count of skipped records.

I will implement the ReAct flow by providing the complete Python code containing the function, assertions to test the scenarios, and printing "ALL TESTS PASSED" at the end.

Action:
```python
import datetime
import math
from collections import defaultdict

def monthly_spend(transactions):

Write Python code for Google Colab that runs a ReAct loop with google.colab.ai.generate_text(). Send the ReAct prompt below to the model and pull the Python code block out of its response. Run the code with exec() inside try/except and capture the output or the full traceback. If it fails, send the error back to the model as an "Observation" and ask it to fix the code. Stop after it succeeds or after 3 attempts. Print every Thought, Code, and Observation, labeled by iteration number.

You are a Python developer using the ReAct method. Work in these stages and label each one:
Thought: Restate the task, list the inputs and outputs, and plan your approach and edge cases.
Action: Write the complete Python code in one ```python block.
Observation: (I will run your code and paste the result or the error here.)
Fix: If the observation shows an error or wrong output, explain the cause, then give corrected code.

Task: Write a function monthly_spend(transactions) that takes a list of dicts with keys "date" (YYYY-MM-DD), "category", and "amount", and returns a dict of total spend per month per category, rounded to 2 decimals.

Requirements:

Python 3, standard library only (no pandas).
Skip records with a missing or invalid date or amount, and count how many were skipped.
Handle an empty list by returning {}.
Include 4 test cases with asserts: normal data, an empty list, a bad date, and a negative refund amount.
Print "ALL TESTS PASSED" at the end.

